# Lasso Regression — Mathematics

**Goal.** Derive Lasso from scratch. Define the L1-norm penalty, write the loss, observe that it is convex but **not differentiable** at any point with a zero coordinate, replace the gradient by the **subgradient**, state the resulting first-order optimality conditions, prove the celebrated **soft-thresholding** identity (the heart of every Lasso algorithm), and connect Lasso to Bayesian MAP estimation under a Laplace prior.

**Role of this notebook.** Pure mathematics — definitions, derivations, theorems. No code, no plots. Intuition is in `01_intuition.ipynb`; algorithms in `03_optimization.ipynb`; implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `01_linear_regression/02_mathematics.ipynb` (OLS gradient, hat matrix, SVD); `03_ridge_regression/02_mathematics.ipynb` (regularised regression in penalty / constraint form, Bayesian view). We will lean heavily on the Ridge derivation and only highlight what changes.

**Stage map.** `01_intuition` → **`02_mathematics`** → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

**Six questions.**

1. What does the L1-norm penalty look like analytically?
2. Why is there no nice closed form like Ridge had?
3. What replaces the gradient when the loss is not smooth?
4. What are the **first-order optimality conditions** for Lasso?
5. What is the **soft-thresholding** operator, and why is it the key formula?
6. What Bayesian prior produces Lasso?

---

**Reading conventions.**

- All vectors are column vectors. Lowercase Latin / Greek = vector; uppercase = matrix.
- Probability distributions are written by name: `Normal(0, $\sigma^2$)`, $\operatorname{Laplace}(0,b)$. No script characters.
- Norms are written with explicit subscripts: `$\|\theta\|_1$` (sum of absolute values), `$\|\theta\|_2$` (Euclidean), `$\|\theta\|_2^2$` (Euclidean, squared).
- Every non-trivial symbol is defined on first use in §0.



## 0. Notation

All symbols used below — defined once. Standard OLS conventions inherited from `01_linear_regression/02_mathematics.ipynb` §0; the Lasso-specific additions are at the bottom of the table.

| Symbol | Type | Meaning |
|---|---|---|
| n | scalar | number of training examples |
| p | scalar | number of features (after the bias trick / centring) |
| X | matrix of size n $\times$ p | design matrix; row i is the i-th example's features |
| y | vector of length n | target vector |
| $\theta$ | vector of length p | model parameters (coefficients) |
| $\theta_j$ | scalar | j-th coordinate of $\theta$ |
| $\|\theta\|_1$ | scalar | **L1 norm** of $\theta$ — the sum of absolute values: $\|\theta\|_1 := |\theta_1| + |\theta_2| + \cdots + |\theta_p|$ |
| $\|\theta\|_2^2$ | scalar | squared Euclidean norm of $\theta$ — sum of squares of the coordinates |
| $\lambda$ | scalar $\ge$ 0 | regularisation strength (same role as in Ridge) |
| $L_{\text{lasso}}(\theta)$ | scalar function | the Lasso loss; see §1.1 below |
| $\hat{\theta}_{\text{lasso}}$ | vector | a minimiser of $L_{\text{lasso}}$ |
| $\text{sign}(z)$ | scalar in $\{-1, 0, 1\}$ | sign of a real number; sign(0) := 0 |
| $\partial\|\theta\|_1$ | set in $\mathbb{R}^p$ | **subdifferential** of the L1 norm at $\theta$; see §3.1 |
| $S_\lambda(z)$ | scalar | **soft-thresholding** operator at threshold $\lambda$; defined in §4.2 |
| t | scalar > 0 | budget in the constrained form (§5) |

**A note on the bias / intercept.** Just like Ridge, the standard practice is to *centre* X and y before fitting, then do not penalise the intercept. The intercept is recovered at the end as $\theta_0 = \operatorname{mean}(y) - \bar{x}^\top\hat{\theta}$. All derivations below assume X and y already centred and skip the intercept; the implementation in `05_hands_on_programming.ipynb` handles it explicitly.



## 1. The Lasso loss

### 1.1 Definition

Starting from the OLS loss (eq. 2.1 of `01_linear_regression/02_mathematics.ipynb`),

> $$L_{\text{lasso}}(\theta) := \frac{1}{n} \|X\theta - y\|_2^2 + \lambda \|\theta\|_1$$   (1.1)

**Lasso regression** is the optimisation problem

> $$\hat{\theta}_{\text{lasso}} \in \arg\min_\theta L_{\text{lasso}}(\theta)$$   (1.2)

("Lasso" stands for *Least Absolute Shrinkage and Selection Operator*, Tibshirani 1996.)

### 1.2 Side-by-side with Ridge

| | Data term | Penalty | Closed form? | Sparse solutions? |
|---|---|---|---|---|
| OLS | $(1/n)\|X\theta - y\|_2^2$ | none | Yes — eq. (4.2) of `01_linear_regression/02_mathematics.ipynb` | No |
| Ridge | $(1/n)\|X\theta - y\|_2^2$ | $\lambda\|\theta\|_2^2$ | Yes — eq. (2.2) of `03_ridge_regression/02_mathematics.ipynb` | No |
| **Lasso** | $(1/n)\|X\theta - y\|_2^2$ | $\lambda\|\theta\|_1$ | **No** | **Yes** |

### 1.3 Convexity

> **Theorem 1.3.** $L_{\text{lasso}}$ is convex on $\mathbb{R}^p$.

**Proof.** The first term is convex (Theorem 3.4 of `01_linear_regression/02_mathematics.ipynb`). The L1 norm is convex because for any $\alpha \in [0,1]$ and any u, v,

```
$$\|\alpha u + (1-\alpha)v\|_1 = \sum_j |\alpha u_j + (1-\alpha)v_j|$$
                     $$\le \sum_j ( \alpha |u_j| + (1-\alpha)|v_j| ) \quad \text{(triangle inequality)}$$
                     $$= \alpha \|u\|_1 + (1-\alpha)\|v\|_1$$
```

Sum of two convex functions is convex. ∎

**Reading.** Convexity guarantees every local minimum is also a global minimum. But unlike Ridge, the Lasso loss is *not* strictly convex (the L1 norm is only piecewise linear), so the minimiser is not necessarily unique. When the minimum is on a face of the L1-ball, every point of that face has the same loss — fortunately this rarely happens with continuous noisy data, and is also harmless: every such point produces the same prediction X $\hat{\theta}$.



## 2. Why there is no closed form

For Ridge, setting the gradient to zero produced eq. (2.2) of `03_ridge_regression/02_mathematics.ipynb`. For Lasso, **the gradient does not exist** at any $\theta$ that has a zero coordinate — and we expect $\hat{\theta}_{\text{lasso}}$ to have many zero coordinates (this was the whole point of using Lasso). So the OLS / Ridge approach breaks down before it begins.

The fix: replace the gradient by an object that *does* exist for non-smooth convex functions — the **subdifferential**. We then need to write down the right generalisation of "set the gradient to zero". The next two sections do exactly that.



## 3. Subgradient calculus, just enough for Lasso

### 3.1 Definition (subgradient and subdifferential)

Let f : $\mathbb{R}^p$ → $\mathbb{R}$ be a convex function. A vector g $\in$ $\mathbb{R}^p$ is a **subgradient** of f at the point x if

> $$f(z) \ge f(x) + g^T (z - x) \quad \text{for every } z \in \mathbb{R}^p$$   (3.1)

The set of all subgradients of f at x is the **subdifferential** $\partial f(x)$.

Reading (3.1) geometrically: a subgradient defines an affine function whose graph lies **below** the graph of f everywhere and **touches** it at the point x. If f is differentiable at x, then $\partial f(x)$ = $\{\nabla f(x)\}$ is a single-point set — the gradient. If f is non-smooth at x, then $\partial f(x)$ typically contains *many* vectors — the subdifferential is the natural generalisation.

### 3.2 Theorem (subdifferential of the L1 norm)

> **Theorem 3.2.** The subdifferential of the L1 norm at a point $\theta \in \mathbb{R}^p$ is the Cartesian product of one-dimensional subdifferentials, one per coordinate:
>
> $$\partial \|\theta\|_1 = \{ g \in \mathbb{R}^p : g_j \in \partial |\theta_j| \ \text{for every } j = 1, \dots, p \}$$
>
> And the one-dimensional subdifferential of the absolute-value function is
>
> $$\partial |z| = \{ \text{sign}(z) \} \quad \text{if } z \ne 0$$
>
> $$\partial |z| = [-1, +1] \quad \text{if } z = 0$$   (3.2)

**Proof.** The L1 norm decomposes as $\|\theta\|_1$ = $\sum_j |\theta_j|$, so its subdifferential factorises coordinate-wise (a standard rule for separable convex functions). For the absolute value:

*Case z > 0.* In a neighbourhood of z, |$\cdot$| coincides with the identity function, which is differentiable with derivative +1. So $\partial$|z| = {+1}.

*Case z < 0.* By symmetry, $\partial$|z| = $\{-1\}$.

*Case z = 0.* Definition (3.1) becomes |w| $\ge$ g $\cdot$ w for every w. For w > 0 we need g $\le$ 1; for w < 0 we need g $\ge$ $-1$. The set of g satisfying both is $[-1,+1]$. ∎

**Reading (3.2) in plain language.** At a non-zero coordinate the L1-norm has a definite slope: +1 if $\theta_j$ > 0, $-1$ if $\theta_j$ < 0. At a zero coordinate the slope is *any* value between $-1$ and +1 — there is a whole interval of valid subgradients.



## 4. First-order optimality and soft-thresholding

### 4.1 Theorem (Lasso first-order condition)

The subgradient version of "gradient equals zero":

> **Theorem 4.1.** A point $\theta^*$ is a minimiser of $L_{\text{lasso}}$ (eq. 1.1) if and only if there exists a subgradient $s \in$ $\partial\|\theta^*\|_1$ such that
>
> $$\frac{2}{n} X^T (X\theta^* - y) + \lambda s = 0$$   (4.1)

**Proof.** A convex function attains its minimum at $\theta^*$ if and only if 0 belongs to its subdifferential at $\theta^*$ (Fermat's rule for convex non-smooth optimisation; standard reference Rockafellar–Wets, *Variational Analysis*, Theorem 10.1). The subdifferential of $L_{\text{lasso}}$ is the sum of the gradient of the smooth part and the subdifferential of the non-smooth part:

```
$\partial L_{\text{lasso}}(\theta)$  =  (2/n) $\cdot$ $X^T (X \theta - y)$   +   $\lambda$ $\cdot$ $\partial\|\theta\|_1$.
```

Setting 0 $\in$ $\partial$ $L_{\text{lasso}}$($\theta^*$) gives (4.1). ∎

**Reading (4.1) coordinate-by-coordinate.** Let $r := X\theta^* - y$ be the residual vector and $x_j$ the j-th column of X. Then (4.1) says: for every j,

- If $\theta_j^*$ > 0, then s_j = +1, so $(2/n)x_j^\top r$ = $-\lambda$.
- If $\theta_j^*$ < 0, then s_j = $-1$, so $(2/n)x_j^\top r$ = $+\lambda$.
- If $\theta_j^*$ = 0, then s_j $\in$ $[-1,+1]$, so $-\lambda$ $\le$ $(2/n)x_j^\top r$ $\le$ $+\lambda$ — equivalently, $|x_j^\top r|$ $\le$ $n\lambda/2$.

The last bullet is the **inactive coordinate condition**: a coefficient is zero at the optimum exactly when the corresponding column of X has small correlation with the residual. Features that are barely correlated with the residual get dropped — this is variable selection, derived from first principles.



### 4.2 Theorem (soft-thresholding — the one-dimensional Lasso closed form)

Consider the simplest possible Lasso problem: one variable, one observation, no design matrix. Solve

> $$\arg\min_\theta \frac{1}{2} (\theta - z)^2 + \lambda |\theta|$$

(Here z is a fixed real number, and we have absorbed the n / 2 factor for cleanliness.)

> **Theorem 4.2.** The unique minimiser is the **soft-thresholding** of z at threshold $\lambda$:
>
> $$S_\lambda(z) := z - \lambda \quad \text{if } z > \lambda$$
>
> $$S_\lambda(z) := z + \lambda \quad \text{if } z < -\lambda$$
>
> $$S_\lambda(z) := 0 \quad \text{if } |z| \le \lambda$$   (4.2)

Equivalently, $S_\lambda(z)=\operatorname{sign}(z)\max(|z|-\lambda,0)$.

**Proof.** Apply Theorem 4.1 to this scalar problem: the data gradient at $\theta$ is $(\theta - z)$, and the optimality condition is

```
0  $\in$  $(\theta - z)$  +  $\lambda\partial|\theta|$.
```

*Case $\theta>0$.* $\partial|\theta|=\{+1\}$, so $\theta-z+\lambda=0$, hence $\theta=z-\lambda$. For this to be positive we need $z>\lambda$.

*Case $\theta$ < 0.* $\partial$|$\theta$| = $\{-1\}$, so $\theta - z - \lambda$ = 0, hence $\theta$ = z $+\lambda$. For this to be negative we need z < $-\lambda$.

*Case $\theta=0$.* $\partial|\theta|=[-1,+1]$, so the condition becomes $0\in -z+\lambda[-1,+1]$, i.e. $z\in[-\lambda,+\lambda]$. This is the entire "dead zone" where the optimum coordinate is zero.

The three cases cover all real z disjointly, so the formula (4.2) is well-defined and unique. ∎

**Reading.** Soft-thresholding has a clean geometric meaning: shrink z toward zero by exactly $\lambda$, and clip to zero anything closer than $\lambda$. The function is continuous, piecewise linear, and has a flat segment around the origin — exactly the source of Lasso's sparsity.

**Why this matters.** Theorem 4.2 is the one-dimensional Lasso problem solved in closed form. The next notebook (`03_optimization.ipynb`) will reduce the *p-dimensional* Lasso problem to a sequence of *one-dimensional* Lasso problems — one per coordinate — by the technique called **coordinate descent**. Each inner step is exactly a soft-thresholding (4.2). So this single formula drives the entire Lasso solver.



## 5. Equivalence to a constrained problem

Just like Ridge (`03_ridge_regression/02_mathematics.ipynb` §5), the Lasso can be written in two equivalent forms — *penalty* and *constraint*.

### 5.1 Theorem (penalty ⇔ constraint)

> **Theorem 5.1.** For every t > 0 there exists a unique $\lambda(t)$ $\ge$ 0 such that
>
> $$\arg\min_\theta \frac{1}{n} \|X\theta - y\|_2^2 \quad \text{subject to} \quad \|\theta\|_1 \le t \quad = \quad \arg\min_\theta L_{\text{lasso}}(\theta; \lambda(t))$$
>
> The map $t \leftrightarrow \lambda(t)$ is monotone decreasing.

**Proof sketch.** Same KKT argument as Ridge (Theorem 5.1 of `03_ridge_regression/02_mathematics.ipynb`): the Lagrangian is the penalised loss with $\lambda$ playing the role of the multiplier; the active constraint condition $\lambda(\|\theta\|_1 - t)$ = 0 fixes $\lambda$ as a decreasing function of t. ∎

### 5.2 Reading

**Geometric picture.** Draw the elliptical contours of the OLS loss centred at $\hat{\theta}_{\text{OLS}}$, and draw the L1-diamond $\|\theta\|_1$ = t. The Lasso solution is the first tangency point as the diamond grows. Because the diamond has *corners on the axes*, the tangency frequently happens *at* a corner — in which case one or more coordinates of $\hat{\theta}_{\text{lasso}}$ are exactly zero. This is the picture from `01_intuition.ipynb` §2, now justified mathematically by Theorem 5.1.

**Contrast with Ridge.** The Ridge ball has no corners, so its tangency point is generic — coordinates are generically non-zero. The difference between Ridge and Lasso is *not* in the depth of regularisation; it is in the *shape* of the constraint set.



## 6. Bayesian interpretation: MAP under a Laplace prior

OLS = MLE under Gaussian noise (`01_linear_regression/02_mathematics.ipynb` Theorem 2.2). Ridge = MAP under Gaussian noise and a Gaussian prior on $\theta$ (`03_ridge_regression/02_mathematics.ipynb` Theorem 6.1). Lasso = MAP under Gaussian noise and a **Laplace** prior on $\theta$.

### 6.1 Definition (the Laplace distribution)

A scalar random variable Z has the **Laplace distribution** with location 0 and scale b > 0, written $Z \sim \operatorname{Laplace}(0,b)$, if its density is

> $$p_{\text{Laplace}}(z; b) = \frac{1}{2b} \exp\left( - \frac{|z|}{b} \right)$$

It is the L1 analogue of the Normal distribution: same shape but with absolute value where Normal has square. The Laplace density is much *sharper* at z = 0 than a Gaussian — it pushes mass toward zero, encoding the prior belief that the parameter is likely small or exactly zero.

### 6.2 Theorem (Lasso as Laplace MAP)

> **Theorem 6.2.** Assume the data model
>
> $$y \mid X, \theta \sim \text{Normal}(X\theta, \sigma^2 I_n)$$
>
> together with the prior
>
> $$\theta_1, \dots, \theta_p \sim \text{Laplace}(0, b) \quad \text{independently}$$
>
> Then the maximum a posteriori estimator of $\theta$ is the Lasso estimator (1.2) with $\lambda$ = 2 $\sigma^2$ / (n b).

**Proof.** Bayes' rule gives `p($\theta$ | y) ∝ p(y | $\theta$) $\cdot$ p($\theta$)`. Take negative log and drop constants:

```
$$-\log p(\theta \mid y) = \frac{1}{2\sigma^2} \|y - X\theta\|_2^2 + \frac{1}{b} \|\theta\|_1 + \text{constant}$$
```

Multiply by $2\sigma^2/n$ and identify the right side as $L_{\text{lasso}}$ evaluated at $\lambda$ = $2\sigma^2/(nb)$. Argmin coincides with argmax of the posterior, giving the claim. ∎

### 6.3 Reading

Reciprocity table:

| Estimator | Likelihood | Prior on $\theta$ |
|---|---|---|
| OLS  | Normal noise | (none / improper flat) |
| Ridge | Normal noise | Normal — $\mathcal{N}(0,\tau^2)$ |
| **Lasso** | Normal noise | **Laplace** — $\operatorname{Laplace}(0,b)$ |

Same recipe: pick a prior, the negative log of the posterior becomes the regularised loss, the MAP estimator becomes the optimum of that loss. The shape of the prior (smooth Gaussian vs. sharp-at-zero Laplace) becomes the shape of the regulariser (L2 vs. L1) which becomes the shape of the constraint region (ball vs. diamond) which becomes the type of solution (dense vs. sparse). Everything in this folder traces back to one design choice — the prior — and unfolds mechanically from there.



## Takeaway

- **Loss (eq. 1.1).** $L_{\text{lasso}}(\theta)=\frac{1}{n}\|X\theta-y\|_2^2+\lambda\|\theta\|_1$. It is convex but not strictly convex, and it is not differentiable wherever some $\theta_j=0$.
- **No closed form.** The gradient does not exist at the typical optimum; we replace it by the subgradient (§3.1).
- **Subdifferential of the L1 norm (Theorem 3.2).** $\partial|z|=\{\operatorname{sign}(z)\}$ for $z\ne0$, and $\partial|0|=[-1,+1]$.
- **First-order condition (Theorem 4.1).** $0\in (2/n)X^\top(X\theta-y)+\lambda\partial\|\theta\|_1$. Coordinate-wise, a feature with small $|x_j^\top r|$ compared to $n\lambda/2$ is set to zero.
- **Soft-thresholding (Theorem 4.2).** $S_\lambda(z)=\operatorname{sign}(z)\max(|z|-\lambda,0)$ solves the one-dimensional Lasso. Every coordinate-descent step reduces to this formula.
- **Constrained equivalent (Theorem 5.1).** Lasso is equivalent to minimizing $\|X\theta-y\|_2^2$ subject to $\|\theta\|_1\le t$. The L1 diamond has corners on the axes, which creates sparse solutions.
- **Bayesian view (Theorem 6.2).** Lasso is MAP under Gaussian likelihood and Laplace prior, with $\lambda=2\sigma^2/(nb)$.

Next: `03_optimization.ipynb` — turn the soft-thresholding formula into coordinate descent and proximal-gradient algorithms.
